# How should purchase revenue be shared across channels?

A visitor can return through several marketing channels before buying. This notebook implements the assessment's rule: assign credit to sessions in the 30 days before each purchase, weighted by the deepest funnel step reached. All records below are synthetic.

## 1. Generate synthetic session events

The session depth is the highest step reached in that session: page view (1), content download (2), add to cart (3), or purchase (4). The purchase session is included. The generated records do not reproduce the assessment dataset or its results.

In [ ]:
import random
import sqlite3
from datetime import datetime, timedelta

random.seed(601)
channels = ('PPC', 'SEO', 'Direct', 'Email')
funnel = ('page_view', 'download', 'add_to_cart', 'purchase')
events = []
sessions = []
event_id = 1
session_id = 1
base_date = datetime(2024, 1, 1)

for user_number in range(400):
    user_id = f'SYN-{user_number:04d}'
    session_days = sorted(random.sample(range(60), random.randint(1, 5)))
    for day in session_days:
        start = base_date + timedelta(days=day, minutes=random.randrange(0, 720))
        channel = random.choice(channels)
        depth = random.choices((1, 2, 3, 4), weights=(0.25, 0.25, 0.25, 0.25))[0]
        sessions.append((session_id, user_id, start.strftime('%Y-%m-%d %H:%M:%S'), channel))
        for step in range(depth):
            event_time = start + timedelta(minutes=step * 7 + random.randrange(0, 3))
            revenue = random.lognormvariate(4.0, 0.7) if funnel[step] == 'purchase' else 0.0
            events.append((event_id, session_id, event_time.strftime('%Y-%m-%d %H:%M:%S'), funnel[step], revenue))
            event_id += 1
        session_id += 1

print(f'Synthetic users: {400:,}')
print(f'Synthetic sessions: {len(sessions):,}')
print(f'Synthetic events: {len(events):,}')
print('Synthetic purchases:', sum(row[3] == 'purchase' for row in events))

## 2. Store users, sessions, and events

The tables separate a visitor's identity, each channel visit, and the actions within that visit. A purchase is an event with a revenue value.

In [ ]:
db = sqlite3.connect(':memory:')
db.executescript('''
CREATE TABLE users (user_id TEXT PRIMARY KEY);
CREATE TABLE sessions (
    session_id INTEGER PRIMARY KEY,
    user_id TEXT NOT NULL REFERENCES users(user_id),
    session_start TEXT NOT NULL,
    channel TEXT NOT NULL
);
CREATE TABLE events (
    event_id INTEGER PRIMARY KEY,
    session_id INTEGER NOT NULL REFERENCES sessions(session_id),
    event_time TEXT NOT NULL,
    event_type TEXT NOT NULL,
    revenue REAL NOT NULL DEFAULT 0
);
''')
db.executemany('INSERT INTO users(user_id) VALUES (?)', [(f'SYN-{i:04d}',) for i in range(400)])
db.executemany('INSERT INTO sessions VALUES (?, ?, ?, ?)', sessions)
db.executemany('INSERT INTO events VALUES (?, ?, ?, ?, ?)', events)
db.commit()
print('Database rows:', db.execute('SELECT (SELECT COUNT(*) FROM users), (SELECT COUNT(*) FROM sessions), (SELECT COUNT(*) FROM events)').fetchone())

## 3. Attribute each purchase across its 30-day path

For each purchase, find sessions from the same visitor that started no more than 30 days earlier. Measure each session's deepest funnel step using events that occurred by the purchase time. Then assign revenue in proportion to session depth.

In [ ]:
attribution_sql = '''
WITH purchases AS (
    SELECT e.event_id AS purchase_id, s.user_id, e.session_id AS purchase_session_id,
           e.event_time AS purchase_time, e.revenue
    FROM events e
    JOIN sessions s ON s.session_id = e.session_id
    WHERE e.event_type = 'purchase'
), touchpoint_depths AS (
    SELECT p.purchase_id, p.purchase_session_id, p.revenue,
           s.session_id, s.channel,
           MAX(CASE e.event_type
               WHEN 'page_view' THEN 1
               WHEN 'download' THEN 2
               WHEN 'add_to_cart' THEN 3
               WHEN 'purchase' THEN 4
               ELSE 0 END) AS depth
    FROM purchases p
    JOIN sessions s ON s.user_id = p.user_id
        AND s.session_start >= datetime(p.purchase_time, '-30 days')
        AND s.session_start <= p.purchase_time
    JOIN events e ON e.session_id = s.session_id
        AND e.event_time <= p.purchase_time
    GROUP BY p.purchase_id, p.purchase_session_id, p.revenue, s.session_id, s.channel
), weighted_touchpoints AS (
    SELECT *, depth * 1.0 / SUM(depth) OVER (PARTITION BY purchase_id) AS credit_share
    FROM touchpoint_depths
    WHERE depth > 0
)
SELECT purchase_id, session_id, channel, depth, credit_share, revenue,
       revenue * credit_share AS attributed_revenue
FROM weighted_touchpoints
ORDER BY purchase_id, session_id
'''
credits = db.execute(attribution_sql).fetchall()
print('Purchase-session credits:', len(credits))
print('First five credits:')
for row in credits[:5]:
    print(row)

## 4. Check conservation and summarize channels

Every attributed purchase must receive 100% of its credit. Across all purchases, allocated revenue must equal eligible purchase revenue.

In [ ]:
purchase_checks = db.execute('''
SELECT purchase_id, SUM(credit_share) AS share_sum,
       MAX(revenue) AS purchase_revenue, SUM(attributed_revenue) AS allocated_revenue
FROM (''' + attribution_sql + ''')
GROUP BY purchase_id
''').fetchall()
assert all(abs(share - 1.0) < 1e-9 for _, share, _, _ in purchase_checks)
assert all(abs(revenue - allocated) < 1e-8 for _, _, revenue, allocated in purchase_checks)

channel_summary = db.execute('''
SELECT channel, SUM(attributed_revenue) AS attributed_revenue,
       SUM(credit_share) AS fractional_purchase_credit
FROM (''' + attribution_sql + ''')
GROUP BY channel ORDER BY attributed_revenue DESC
''').fetchall()
source_revenue = db.execute("SELECT SUM(revenue) FROM events WHERE event_type='purchase'").fetchone()[0]
attributed_revenue = sum(row[1] for row in channel_summary)
print('Purchase paths checked:', len(purchase_checks))
print(f'Purchase revenue: ${source_revenue:,.2f}')
print(f'Attributed revenue: ${attributed_revenue:,.2f}')
print('Revenue by channel:')
for row in channel_summary:
    print(f'{row[0]}: ${row[1]:,.2f} revenue credit across {row[2]:.2f} fractional purchases')

## Interpretation and limits

This rule assigns more reported credit to sessions that reached deeper funnel steps. It does not estimate what would have happened without a channel. Use a randomized lift test or another causal design to estimate incremental channel impact.

The assessment does not include campaign cost. Add cost by channel and campaign before using these credits to calculate return on ad spend. Agree on the 30-day window, treatment of direct traffic, and inclusion of the purchase session with marketing before using the report.

## Research

- [Google Analytics: Attribution](https://support.google.com/analytics/answer/10596866)
- [Gordon et al., Marketing Science: A Comparison of Approaches to Advertising Measurement](https://pubsonline.informs.org/doi/10.1287/mksc.2018.1135)
